# Legal-form coverage

Which legal forms are recorded each year, and where does coverage change?

**Scope:** All cleaned recipient-year records for 2020-2025, before the comparative cohort filter. Summaries use legal-form/year groups and unique recipient codes; 2025 is the reference year for the later cohort.

This notebook describes the legal forms present in the cleaned data and changes in their recorded coverage. Absence from a source year is not proof that a legal form became ineligible. The comparative analyses use the explicit population rule in the methodology page.

The [Methodology](../METHODOLOGY.md) documents definitions and correction policies. The [Project overview](../README.md) describes the notebook sequence and dataset rebuild process.

## Standardized legal-form labels

The analysis uses cleaned funding records and their standardized legal-form field.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Locate the checkout from the project folder or a notebook subfolder.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "analysis_helpers.py").is_file()
     and (p / "config" / "sources.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project folder.")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chart_style import apply_chart_style, colors_from_style, growth_cmap_from_style, load_chart_style

chart_style = load_chart_style(PROJECT_ROOT)
apply_chart_style(chart_style)
CHART_COLORS = colors_from_style(chart_style)
GROWTH_CMAP = growth_cmap_from_style(chart_style)
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

vmi = pd.read_csv(INTERIM_DIR / "vmi_support_by_recipient.csv", low_memory=False, dtype={"recipient_code": "string"})

vmi["tax_year"] = pd.to_numeric(vmi["tax_year"], errors="coerce")
vmi["recipient_code"] = vmi["recipient_code"].astype("string").str.strip()
vmi["requests"] = pd.to_numeric(vmi["requests"], errors="coerce")
vmi["calculated_pmd_eur"] = pd.to_numeric(vmi["calculated_pmd_eur"], errors="coerce")
vmi["legal_form_clean"] = vmi["legal_form"].fillna("Unknown").str.strip()

vmi.head()



,tax_year,recipient_municipality,recipient_code,recipient_name,legal_form,requests,calculated_pmd_eur,legal_form_clean
0,2025,Kauno m. sav.,191678556,Socialinių inovacijų fondas,Labdaros ir paramos fondas,7,604.80,Labdaros ir paramos fondas
1,2025,Plungės r. sav.,170075590,"Sporto klubas ""Makabi""",Asociacija,4,192.18,Asociacija
2,2025,Kauno m. sav.,193158860,Moters užimtumo informacijos centras,Asociacija,4,316.10,Asociacija
3,2025,Vilniaus m. sav.,124467262,"Paramos fondas ""Švietimo ir tyrimų institutas""",Labdaros ir paramos fondas,19,985.02,Labdaros ir paramos fondas
4,2025,Jurbarko r. sav.,191250833,Skirsnemunės Šv. Jurgio parapija,Tradicinė religinė bendruomenė ar bendrija,9,788.37,Tradicinė religinė bendruomenė ar bendrija


## Annual legal-form and recipient counts

Counts distinguish legal-form categories, unique recipient codes and recipient records. Differences in category coverage help explain why the project reports cohort exclusions explicitly.

In [2]:
legal_forms_per_year = (
    vmi.groupby("tax_year")
      .agg(
          number_of_legal_forms=("legal_form_clean", lambda x: x.nunique(dropna=True)),
          number_of_recipients=("recipient_code", "nunique"),
          recipient_records=("recipient_code", "count")
      )
      .reset_index()
)

legal_forms_per_year


,tax_year,number_of_legal_forms,number_of_recipients,recipient_records
0,2020,14,23240,23240
1,2021,15,22798,22798
2,2022,15,22147,22147
3,2023,17,22555,22555
4,2024,12,14511,14511
5,2025,9,15483,15483


## The 2025 reference population

This table ranks legal forms by their number of recipients in 2025 and includes their requests and funding. Presence in this table defines a data-based reference, not a legal assessment.

In [3]:
legal_forms_2025_summary = (
    vmi[vmi["tax_year"] == 2025]
      .groupby("legal_form_clean")
      .agg(
          recipients=("recipient_code", "nunique"),
          requests=("requests", "sum"),
          calculated_pmd_eur=("calculated_pmd_eur", "sum")
      )
      .reset_index()
      .sort_values("recipients", ascending=False)
)

legal_forms_2025_summary


,legal_form_clean,recipients,requests,calculated_pmd_eur
0,Asociacija,8331,173935,10881416.23
8,Viešoji įstaiga,5101,237308,16102361.54
1,Labdaros ir paramos fondas,1030,111455,7177896.66
4,Profesinė sąjunga ar susivienijimas,662,59677,2539275.59
6,Tradicinė religinė bendruomenė ar bendrija,216,8875,548828.00
5,Religinė bendruomenė ar bendrija,76,1201,64554.23
2,Politinis komitetas,35,2889,119087.11
3,Politinė partija,23,56752,2689274.27
7,"Užsienyje įsteigta lietuvių bendruomenė, kita ...",9,27,1931.11


## Legal forms absent in 2024 or 2025

The first set contains forms seen before 2024 but absent in 2024. The second contains forms seen before 2024, present in 2024 and absent in 2025. This is not a complete list of every transition: it does not describe reappearances or all newly observed forms.

In [4]:
legal_forms_before_2024 = set(vmi.loc[vmi["tax_year"] < 2024, "legal_form_clean"].unique())
legal_forms_2024 = set(vmi.loc[vmi["tax_year"] == 2024, "legal_form_clean"].unique())
legal_forms_2025 = set(vmi.loc[vmi["tax_year"] == 2025, "legal_form_clean"].unique())

disappeared_2024 = sorted(legal_forms_before_2024 - legal_forms_2024)
disappeared_2025 = sorted((legal_forms_before_2024 - legal_forms_2025) - set(disappeared_2024))

disappeared_table = pd.DataFrame({
    "tax_year": [2024] * len(disappeared_2024) + [2025] * len(disappeared_2025),
    "disappeared_legal_form": disappeared_2024 + disappeared_2025
})

disappeared_table


,tax_year,disappeared_legal_form
0,2024,Biudžetinė įstaiga
1,2024,"Prekybos, pramonės ir amatų rūmai"
2,2024,Regiono plėtros taryba
3,2024,Sodininkų bendrija
4,2024,Šeimyna
5,2025,Bendrija
6,2025,Mažoji bendrija
7,2025,Uždaroji akcinė bendrovė


## Legal-form labels containing 'bendrija'

This text filter can include several legal-form labels. The table retains each label separately rather than treating them as one legal category.

In [5]:
bendrija_df = vmi[vmi["legal_form_clean"].str.contains("bendrija", case=False, na=False)].copy()

bendrija_by_year = (
    bendrija_df.groupby(["tax_year", "legal_form_clean"])
      .agg(
          recipients=("recipient_code", "nunique"),
          requests=("requests", "sum"),
          calculated_pmd_eur=("calculated_pmd_eur", "sum")
      )
      .reset_index()
      .sort_values(["tax_year", "calculated_pmd_eur"], ascending=[True, False])
)

bendrija_by_year


,tax_year,legal_form_clean,recipients,requests,calculated_pmd_eur
4,2020,Tradicinė religinė bendruomenė ar bendrija,912,28599,1034668.51
0,2020,Bendrija,3419,20556,946477.64
3,2020,Sodininkų bendrija,690,6221,266200.59
2,2020,Religinė bendruomenė ar bendrija,113,2266,67302.27
1,2020,Mažoji bendrija,11,66,2179.52
9,2021,Tradicinė religinė bendruomenė ar bendrija,922,28186,1162021.37
5,2021,Bendrija,3437,20000,1040952.84
8,2021,Sodininkų bendrija,711,6678,315972.52
7,2021,Religinė bendruomenė ar bendrija,116,2211,75562.61
6,2021,Mažoji bendrija,14,80,3702.24


## Annual coverage of a specific legal form

This exact-label filter summarizes the recorded family-based care legal form by year. A missing year in the output means no matching records, not an automatically assigned zero.

In [6]:
seimyna_by_year = (
    vmi[vmi["legal_form_clean"].eq("Šeimyna")]
      .groupby("tax_year")
      .agg(
          recipients=("recipient_code", "nunique"),
          requests=("requests", "sum"),
          calculated_pmd_eur=("calculated_pmd_eur", "sum")
      )
      .reset_index()
)

seimyna_by_year


,tax_year,recipients,requests,calculated_pmd_eur
0,2020,38,339,18802.97
1,2021,32,311,14647.55
2,2022,28,238,10643.67
3,2023,30,243,11250.90


## Reading the results together

Coverage differences justify an explicit, shared comparison population. They do not establish the legal reasons for a category's absence.

[Previous notebook](02_pmd_distribution.ipynb) | [Next notebook](04_top_1_percent_overview.ipynb) | [Reading guide](../README.md#reading-order)